<a href="https://colab.research.google.com/github/Ololade117/Inducing-Polysemanticity/blob/main/Joint_Scaling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Joint Scaling: Data + Parameters + Training Time Together

Unlike the earlier sweep (which varied params OR steps while holding the
other fixed), this scales **all three together** -- width, training steps,
and the amount of unique training data -- across 10 trials, for both
`normal` and `flex` bias, starting from the same ~3.7M-param baseline.

**New data source**: Tiny Shakespeare (1.1M characters) can't support the
larger trials without heavy repetition, so this uses the *Folger Shakespeare*
complete-works dataset instead (42 plays and poems, ~5.3M characters after
cleanup -- about 5x more text). The tokenizer is retrained from scratch on
this corpus (same chunked + incremental-counting BPE as before).

**How the three dimensions scale together**: geometric spacing from 1x to
~5.9x (the data axis is the hard ceiling -- that's as far as it can stretch
before the largest trial would need to repeat text), applied consistently to
steps and data, and to `d_model` (so params scale by a comparable, if
slightly larger, factor -- ~9.5x by the last trial, since `d_model` is
quantized to multiples of 64 for clean attention heads).

| trial | d_model | params | steps | data (chars) |
|---|---|---|---|---|
| 1 (baseline) | 256 | 3.7M | 15,000 | 900,000 |
| ... | ... | ... | ... | ... |
| 10 | 832 | 35.0M | 89,000 | 5,330,000 (full corpus) |

**Cost**: rough per-trial compute scales like params x steps, so the last
trial costs roughly params_ratio x steps_ratio ~= 9.5 x 5.9 ~= 56x the
baseline -- similar total order of magnitude to the earlier params-only
sweep, just now with data scaling alongside it instead of held fixed. Same
resume-safety approach as before: every run checks the Hub before training
and skips (reloading just to record its dev loss) anything already pushed.


## 0. Setup


In [ ]:
# If a package is missing, uncomment and run:
# !pip install -q torch huggingface_hub matplotlib

import urllib.request
from collections import Counter

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from huggingface_hub import PyTorchModelHubMixin, notebook_login

torch.manual_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

HF_USERNAME = "Ololade117"   # <-- replace with your Hugging Face username
PUSH_TO_HUB = True                 # set False to skip the upload steps while testing

# Run this once, then paste a Hugging Face token with write access when prompted.
from huggingface_hub import notebook_login
notebook_login()


Using device: cuda


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 1. Data: the Folger Shakespeare complete works

Downloaded as a repo archive (42 individual play/poem files), concatenated,
CRLF-normalized, and cleaned the same way as before -- drop near-singleton
characters (here: a handful of stray `<`, `>`, `&`, and one curly-apostrophe
encoding slip, 12 occurrences total out of 5.3M characters), keep everything
else including genuinely rare-but-real characters like digits and `X`/`Q`/`Z`.


In [ ]:
import tarfile, io

url = "https://codeload.github.com/cobanov/shakespeare-dataset/tar.gz/refs/heads/main"
raw = urllib.request.urlopen(url).read()

text_parts = []
with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tf:
    names = sorted(m.name for m in tf.getmembers() if m.name.endswith(".txt") and "/text/" in m.name)
    for name in names:
        text_parts.append(tf.extractfile(name).read().decode("utf-8", errors="replace"))

text = "\n\n".join(text_parts).replace("\r\n", "\n").replace("\r", "\n")
print(f"{len(names)} files, {len(text):,} characters combined")

MIN_CHAR_COUNT = 30
counts = Counter(text)
dropped = {ch for ch, n in counts.items() if n < MIN_CHAR_COUNT}
print(f"dropping {len(dropped)} near-singleton characters {sorted(dropped)!r} "
      f"({sum(counts[c] for c in dropped)} occurrences out of {len(text):,})")
text = "".join(ch for ch in text if ch not in dropped)

n1 = int(0.8 * len(text))
n2 = int(0.9 * len(text))
train_text, val_text, test_text = text[:n1], text[n1:n2], text[n2:]
print(f"train/val/test characters: {len(train_text):,} / {len(val_text):,} / {len(test_text):,}")


42 files, 5,330,418 characters combined
dropping 4 near-singleton characters ['&', '<', '>', '’'] (12 occurrences out of 5,330,418)
train/val/test characters: 4,264,324 / 533,041 / 533,041


## 2. Tokenizer

Same `BPETokenizer` class as before (regex chunking + incremental counting).
Trained on the first 900K characters of `train_text` rather than the full
5.3M -- BPE merge statistics stabilize well before that, and it keeps
training time comparable to before (~90-120s) rather than ~5x longer.
The resulting tokenizer then encodes the *entire* corpus for actual use.


In [ ]:
import re

GPT2_SPLIT_PATTERN = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[a-zA-Z]+| ?[0-9]+| ?[^\sa-zA-Z0-9]+|\s+(?!\S)|\s+"""


class BPETokenizer:
    """
    Byte-level BPE tokenizer, trained from scratch -- the same core algorithm
    behind GPT-2/GPT-4's tokenizers, including their regex pre-splitting:
    text is chunked into word-like pieces first, and merges never cross a
    chunk boundary. That's what lets a small custom vocabulary converge on
    whole words instead of getting stuck on arbitrary byte pairs.
    """

    def __init__(self):
        self.merges = {}                                  # (id1, id2) -> new_id, in learn order
        self.vocab = {i: bytes([i]) for i in range(256)}   # id -> raw bytes
        self.pattern = re.compile(GPT2_SPLIT_PATTERN)

    @property
    def vocab_size(self):
        return len(self.vocab)

    @staticmethod
    def _merge(ids, pair, new_id):
        new_ids, i, n = [], 0, len(ids)
        while i < n:
            if i < n - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
                new_ids.append(new_id)
                i += 2
            else:
                new_ids.append(ids[i])
                i += 1
        return new_ids

    @staticmethod
    def _merge_and_update_counts(ids, pair, new_id, counts):
        """Same merge as `_merge`, but updates `counts` in place for exactly
        the pairs that changed, instead of recounting everything from scratch."""
        new_ids, i, n = [], 0, len(ids)
        while i < n:
            if i < n - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
                if new_ids:
                    left = new_ids[-1]
                    counts[(left, pair[0])] = counts.get((left, pair[0]), 0) - 1
                    counts[(left, new_id)] = counts.get((left, new_id), 0) + 1
                if i + 2 < n:
                    right = ids[i + 2]
                    counts[(pair[1], right)] = counts.get((pair[1], right), 0) - 1
                    counts[(new_id, right)] = counts.get((new_id, right), 0) + 1
                new_ids.append(new_id)
                i += 2
            else:
                new_ids.append(ids[i])
                i += 1
        return new_ids

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256, "vocab_size must cover at least the raw byte values"
        chunks = [list(chunk.encode("utf-8")) for chunk in self.pattern.findall(text)]

        counts = {}
        for ids in chunks:
            for a, b in zip(ids, ids[1:]):
                counts[(a, b)] = counts.get((a, b), 0) + 1

        num_merges = vocab_size - 256
        for i in range(num_merges):
            counts = {k: v for k, v in counts.items() if v > 0}
            if not counts:
                break
            pair = max(counts, key=counts.get)
            new_id = 256 + i
            chunks = [self._merge_and_update_counts(ids, pair, new_id, counts) for ids in chunks]
            counts.pop(pair, None)
            self.merges[pair] = new_id
            self.vocab[new_id] = self.vocab[pair[0]] + self.vocab[pair[1]]
            if verbose and (i + 1) % 128 == 0:
                print(f"  merge {i+1:4d}/{num_merges}: {pair} -> {new_id}  {self.vocab[new_id]!r}")

    def encode(self, text):
        ids = []
        for chunk in self.pattern.findall(text):
            chunk_ids = list(chunk.encode("utf-8"))
            for pair, new_id in self.merges.items():  # replay merges in the order learned
                chunk_ids = self._merge(chunk_ids, pair, new_id)
            ids.extend(chunk_ids)
        return ids

    def decode(self, ids):
        raw = b"".join(self.vocab[i] for i in ids)
        return raw.decode("utf-8", errors="replace")




In [ ]:
TOKENIZER_VOCAB_SIZE = 1024
TOKENIZER_TRAIN_CHARS = 900_000

tokenizer = BPETokenizer()
tokenizer.train(train_text[:TOKENIZER_TRAIN_CHARS], vocab_size=TOKENIZER_VOCAB_SIZE, verbose=True)

train_data_full = torch.tensor(tokenizer.encode(train_text), dtype=torch.long)
val_data = torch.tensor(tokenizer.encode(val_text), dtype=torch.long)
test_data = torch.tensor(tokenizer.encode(test_text), dtype=torch.long)
VOCAB_SIZE = tokenizer.vocab_size

print(f"\nvocab_size = {VOCAB_SIZE} (256 base bytes + {len(tokenizer.merges)} merges)")
print(f"train/val/test tokens: {len(train_data_full):,} / {len(val_data):,} / {len(test_data):,}")
print(f"compression: {len(train_text) / len(train_data_full):.2f} characters per token")

sample = val_text[1000:1200]
roundtrip = tokenizer.decode(tokenizer.encode(sample))
assert roundtrip == sample, "tokenizer round-trip failed -- stop and debug this before training"
print("OK: decode(encode(x)) == x")


  merge  128/768: (97, 261) -> 383  b'ain'
  merge  256/768: (397, 297) -> 511  b' love'
  merge  384/768: (575, 83) -> 639  b'ROS'
  merge  512/768: (473, 364) -> 767  b' never'
  merge  640/768: (303, 784) -> 895  b' gentle'
  merge  768/768: (570, 309) -> 1023  b'ople'

vocab_size = 1024 (256 base bytes + 768 merges)
train/val/test tokens: 1,838,432 / 232,145 / 230,236
compression: 2.32 characters per token
OK: decode(encode(x)) == x


## 3. Architecture

Unchanged from the training notebook -- `ManualLinear`, `CausalSelfAttention`,
`TransformerBlock`, `NormalBiasNet`.


In [ ]:
class ManualLinear(nn.Module):
    """
    One h = h @ W + b step. `bias_type` selects which of the six bias
    mechanisms this layer uses -- every mechanism is defined right here.
    """

    def __init__(self, in_dim, out_dim, bias_type="normal", block_size=None):
        super().__init__()
        self.bias_type = bias_type
        self.weight = nn.Parameter(torch.randn(in_dim, out_dim) * in_dim ** -0.5)

        if bias_type == "normal":
            self.bias = nn.Parameter(torch.zeros(out_dim))

        elif bias_type in ("gated_sigmoid", "gated_relu", "gated_tanh"):
            self.bias = nn.Parameter(torch.zeros(out_dim))
            self.gate_param = nn.Parameter(torch.zeros(out_dim))  # learned, pre-activation

        elif bias_type == "per_position":
            assert block_size is not None, "per_position bias needs block_size"
            self.bias = nn.Parameter(torch.zeros(block_size, out_dim))

        elif bias_type == "flex":
            # maps h itself down to one scalar per row (per example, per position)
            self.flex_vec = nn.Parameter(torch.zeros(in_dim, 1))

        else:
            raise ValueError(f"Unknown bias_type: {bias_type!r}")

    def forward(self, h):
        out = h @ self.weight

        if self.bias_type == "normal":
            return out + self.bias

        elif self.bias_type == "gated_sigmoid":
            return out + torch.sigmoid(self.gate_param) * self.bias

        elif self.bias_type == "gated_relu":
            return out + torch.relu(self.gate_param) * self.bias

        elif self.bias_type == "gated_tanh":
            return out + torch.tanh(self.gate_param) * self.bias

        elif self.bias_type == "per_position":
            # self.bias: (block_size, out_dim) broadcasts over the batch dim
            return out + self.bias

        elif self.bias_type == "flex":
            flex_bias = h @ self.flex_vec  # (..., 1) -- one value per row
            return out + flex_bias

        else:
            raise ValueError(f"Unknown bias_type: {self.bias_type!r}")


class CausalSelfAttention(nn.Module):
    """
    Standard multi-head causal self-attention. Plain nn.Linear projections,
    deliberately NOT varied by bias_type -- isolating the bias experiment to
    the MLP sublayer, same reasoning as the output head staying "normal".
    """

    def __init__(self, d_model, n_head, block_size):
        super().__init__()
        assert d_model % n_head == 0, "d_model must be divisible by n_head"
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, nh, T, hd)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)   # handles the causal mask
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)


class TransformerBlock(nn.Module):
    """
    x = x + attn(ln(x));  x = x + mlp(ln(x)) -- the bias experiment lives
    entirely inside the MLP's two ManualLinear layers.
    """

    def __init__(self, d_model, n_head, block_size, bias_name):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, block_size)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp_in = ManualLinear(d_model, 4 * d_model, bias_type=bias_name, block_size=block_size)
        self.mlp_out = ManualLinear(4 * d_model, d_model, bias_type=bias_name, block_size=block_size)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        h = F.gelu(self.mlp_in(self.ln2(x)))
        x = x + self.mlp_out(h)
        return x


class NormalBiasNet(
    nn.Module,
    PyTorchModelHubMixin,
    license="mit",
):
    """
    A small transformer. `bias_name` picks the bias mechanism used by every
    MLP sublayer (see ManualLinear/TransformerBlock above) -- attention, layer
    norms, and the output head stay identical across all six networks built
    later in this notebook.
    """

    def __init__(self, vocab_size, hidden_dim, n_head, block_size, n_layers=3, bias_name="normal"):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim  # this is d_model -- the one dimension carried through the residual stream
        self.n_head = n_head
        self.block_size = block_size
        self.n_layers = n_layers
        self.bias_name = bias_name

        self.token_emb = nn.Parameter(torch.randn(vocab_size, hidden_dim) * 0.02)
        self.pos_emb = nn.Parameter(torch.randn(block_size, hidden_dim) * 0.02)

        self.blocks = nn.ModuleList([
            TransformerBlock(hidden_dim, n_head, block_size, bias_name) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(hidden_dim)

        # The output head always uses a plain bias -- same reasoning as before.
        self.out = ManualLinear(hidden_dim, vocab_size, bias_type="normal")

    def forward(self, X):
        B, T = X.shape
        x = self.token_emb[X] + self.pos_emb[:T]   # (batch, block_size, hidden_dim)

        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)

        x = x[:, -1, :]           # attention already mixed in every earlier position
        logits = self.out(x)      # raw logits -- loss fn applies softmax
        return logits




In [ ]:
BLOCK_SIZE = 6
N_LAYERS = 4
BATCH_SIZE = 256
LR = 2e-3
HEAD_DIM = 64  # n_head = d_model // HEAD_DIM, kept fixed as d_model scales


## 4. Training / evaluation / generation / push helpers

Same as before, except `get_batch` now takes an explicit `data` slice (each
trial trains on its own prefix of `train_data_full`, not the whole thing),
and `push` names repos with all three scaled dimensions.


In [ ]:
trained_models = {}

def get_batch(data, block_size=BLOCK_SIZE, batch_size=BATCH_SIZE):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    idx = ix.unsqueeze(1) + torch.arange(block_size)
    return data[idx], data[ix + block_size]


@torch.no_grad()
def estimate_loss(model, data, eval_iters=50):
    model.eval()
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
        Xb, Yb = get_batch(data)
        logits = model(Xb.to(DEVICE))
        losses[k] = F.cross_entropy(logits, Yb.to(DEVICE)).item()
    model.train()
    return losses.mean().item()


def train_model(model, train_data, steps, batch_size=BATCH_SIZE, lr=LR, log_every=None):
    log_every = log_every or max(steps // 10, 1)
    model.to(DEVICE).train()
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=max(steps // 3, 1), gamma=0.3)

    history = {"step": [], "train_loss": [], "dev_loss": []}
    for step in range(1, steps + 1):
        Xb, Yb = get_batch(train_data, batch_size=batch_size)
        Xb, Yb = Xb.to(DEVICE), Yb.to(DEVICE)
        logits = model(Xb)
        loss = F.cross_entropy(logits, Yb)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        if step % log_every == 0 or step == 1:
            dev_loss = estimate_loss(model, val_data)
            history["step"].append(step)
            history["train_loss"].append(loss.item())
            history["dev_loss"].append(dev_loss)
            print(f"  step {step:6d} | train {loss.item():.3f} | dev {dev_loss:.3f}")

    return history


@torch.no_grad()
def generate(model, prompt="ROMEO:", length=60, block_size=BLOCK_SIZE):
    model.eval()
    pad_id = 32
    ids = tokenizer.encode(prompt)
    if len(ids) < block_size:
        ids = [pad_id] * (block_size - len(ids)) + ids
    context = ids[-block_size:]
    generated = list(ids)
    for _ in range(length):
        x = torch.tensor([context]).to(DEVICE)
        probs = torch.softmax(model(x), dim=-1)
        ix = torch.multinomial(probs, num_samples=1).item()
        generated.append(ix)
        context = context[1:] + [ix]
    model.train()
    return tokenizer.decode(generated)


## 5. Joint scaling grid and config table


In [ ]:
BASELINE_D_MODEL = 256
BASELINE_STEPS = 15000
BASELINE_DATA_CHARS = 900_000
FULL_CORPUS_CHARS = len(train_text)

def total_params_estimate(d_model, bias_type="normal"):
    n_head = d_model // HEAD_DIM
    token_emb, pos_emb = VOCAB_SIZE * d_model, BLOCK_SIZE * d_model
    per_block = (d_model*3*d_model + 3*d_model + d_model*d_model + d_model + 2*d_model + 2*d_model
                 + d_model*(4*d_model) + 4*d_model + (4*d_model)*d_model + d_model)
    return token_emb + pos_emb + N_LAYERS*per_block + 2*d_model + d_model*VOCAB_SIZE + VOCAB_SIZE

SCALE_MAX = FULL_CORPUS_CHARS / BASELINE_DATA_CHARS
baseline_params = total_params_estimate(BASELINE_D_MODEL)

d_models = [BASELINE_D_MODEL]
target_scales = [SCALE_MAX ** (i / 9) for i in range(1, 10)]
for target in target_scales:
    target_params = baseline_params * target
    cand = d_models[-1] + HEAD_DIM
    while total_params_estimate(cand) < target_params:
        cand += HEAD_DIM
    d_models.append(max(cand, d_models[-1] + HEAD_DIM))

steps_grid = [BASELINE_STEPS] + [round(BASELINE_STEPS * s / 1000) * 1000 for s in target_scales]
data_grid = [BASELINE_DATA_CHARS] + [round(min(FULL_CORPUS_CHARS, BASELINE_DATA_CHARS * s)) for s in target_scales]
for i in range(1, 10):
    if steps_grid[i] <= steps_grid[i - 1]:
        steps_grid[i] = steps_grid[i - 1] + 1000
    if data_grid[i] <= data_grid[i - 1]:
        data_grid[i] = min(FULL_CORPUS_CHARS, data_grid[i - 1] + 50_000)

print(f"{'trial':>5s} {'d_model':>8s} {'params_est':>12s} {'steps':>8s} {'data_chars':>11s}")
for i in range(10):
    print(f"{i+1:>5d} {d_models[i]:>8d} {total_params_estimate(d_models[i]):>12,d} "
          f"{steps_grid[i]:>8,d} {data_grid[i]:>11,d}")

BIAS_TYPES = ["normal", "flex"]
configs = []
for bias_type in BIAS_TYPES:
    for i in range(10):
        configs.append({"bias_type": bias_type, "trial": i + 1, "d_model": d_models[i],
                         "n_head": d_models[i] // HEAD_DIM, "steps": steps_grid[i],
                         "data_chars": data_grid[i]})
print(f"\n{len(configs)} total configs ({len(configs)//2} per bias type)")


trial  d_model   params_est    steps  data_chars
    1      256    3,686,400   15,000     900,000
    2      320    5,590,784   18,000   1,069,818
    3      384    7,888,384   21,000   1,271,679
    4      448   10,579,200   25,000   1,511,628
    5      512   13,663,232   30,000   1,796,853
    6      576   17,140,480   36,000   2,135,896
    7      640   21,010,944   42,000   2,538,912
    8      704   25,274,624   50,000   3,017,972
    9      768   29,931,520   60,000   3,587,424
   10      832   34,981,632   71,000   4,264,324

20 total configs (10 per bias type)


## 6. Run function (Hub-resume-safe) and main loop

Each trial's training data is a prefix of `train_data_full`, sized to match
that trial's `data_chars` budget (converted to a token count via the
tokenizer's measured compression ratio).


In [ ]:
from huggingface_hub import HfApi
_api = HfApi()

def repo_already_pushed(repo_id):
    try:
        return _api.repo_exists(repo_id)
    except Exception:
        return False


CHARS_PER_TOKEN = len(train_text) / len(train_data_full)

def run_one(config, skip_if_pushed=True):
    bias_type, trial = config["bias_type"], config["trial"]
    d_model, n_head, steps = config["d_model"], config["n_head"], config["steps"]
    data_tokens = min(len(train_data_full), round(config["data_chars"] / CHARS_PER_TOKEN))
    trial_train_data = train_data_full[:data_tokens]

    torch.manual_seed(42)
    model = NormalBiasNet(vocab_size=VOCAB_SIZE, hidden_dim=d_model, n_head=n_head,
                           block_size=BLOCK_SIZE, n_layers=N_LAYERS, bias_name=bias_type)
    n_params = sum(p.numel() for p in model.parameters())
    run_name = f"{bias_type}-t{trial}-{n_params/1e6:.1f}M-{steps}steps-{data_tokens}tok"
    repo_id = f"{HF_USERNAME}/jointscale-{run_name}"

    if skip_if_pushed and PUSH_TO_HUB and repo_already_pushed(repo_id):
        print(f"trial {trial:2d} [{bias_type:6s}] {run_name}: already pushed -- loading instead")
        loaded = NormalBiasNet.from_pretrained(
            repo_id, vocab_size=VOCAB_SIZE, hidden_dim=d_model, n_head=n_head,
            block_size=BLOCK_SIZE, n_layers=N_LAYERS, bias_name=bias_type,
        ).to(DEVICE).eval()
        dev_loss = estimate_loss(loaded, val_data)
        sample = generate(loaded, prompt="ROMEO:", length=60)
        return {**config, "run_name": run_name, "repo_id": repo_id, "params": n_params,
                "final_dev_loss": dev_loss, "history": None, "sample": sample, "skipped": True}

    print(f"\ntrial {trial:2d} [{bias_type:6s}] === {run_name} === "
          f"(d_model={d_model} steps={steps} data_tokens={data_tokens:,} params={n_params:,})")

    history = train_model(model, trial_train_data, steps=steps)
    sample = generate(model, prompt="ROMEO:", length=60)

    if PUSH_TO_HUB:
        model.push_to_hub(repo_id)
        print(f"Pushed to https://huggingface.co/{repo_id}")

    return {**config, "run_name": run_name, "repo_id": repo_id, "params": n_params,
            "final_dev_loss": history["dev_loss"][-1], "history": history, "sample": sample,
            "skipped": False}


In [ ]:
results = []
for i, config in enumerate(configs):
    print(f"\n########## run {i + 1}/{len(configs)} ##########")
    results.append(run_one(config))

print(f"\nDone: {len(results)} results "
      f"({sum(r['skipped'] for r in results)} loaded, {sum(not r['skipped'] for r in results)} trained).")



########## run 1/20 ##########
trial  1 [normal] normal-t1-3.7M-15000steps-388007tok: already pushed -- loading instead


config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/14.8M [00:00<?, ?B/s]


########## run 2/20 ##########
trial  2 [normal] normal-t2-5.6M-18000steps-461219tok: already pushed -- loading instead


config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/14.8M [00:00<?, ?B/s]


########## run 12/20 ##########
trial  2 [flex  ] flex-t2-5.6M-18000steps-461219tok: already pushed -- loading instead


config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/22.4M [00:00<?, ?B/s]

## 7. Compare

Dev loss vs. trial index (1 = baseline, 10 = largest) -- since params, steps,
and data all move together here, trial index is the cleanest shared x-axis;
the params/steps/data table from section 5 gives the actual values each
point corresponds to.


In [ ]:
plt.figure(figsize=(7, 4.5))
for bias_type in BIAS_TYPES:
    rows = sorted([r for r in results if r["bias_type"] == bias_type], key=lambda r: r["trial"])
    plt.plot([r["trial"] for r in rows], [r["final_dev_loss"] for r in rows], marker="o", label=bias_type)
plt.xlabel("trial (1=baseline ~3.7M/15k steps/900K chars, 10=largest)")
plt.ylabel("final dev loss")
plt.title("Joint scaling: params + steps + data increasing together")
plt.legend()
plt.tight_layout()
plt.show()

print(f"{'bias type':10s} {'trial':>6s} {'params':>12s} {'steps':>8s} {'data_chars':>11s} {'dev loss':>9s}")
for bias_type in BIAS_TYPES:
    rows = sorted([r for r in results if r["bias_type"] == bias_type], key=lambda r: r["trial"])
    for r in rows:
        print(f"{bias_type:10s} {r['trial']:>6d} {r['params']:>12,d} {r['steps']:>8,d} "
              f"{r['data_chars']:>11,d} {r['final_dev_loss']:>9.3f}")
